# Explore a trained model

Load a finished run, feed it hand-made operator inputs and compare with the analytical
solution. Kernel: the conda env `pinobench` (needs `ipykernel`: `pip install -e ".[notebook]"`). Plots appear inline (no window needed over SSH).

Inputs are given in the problem's own parametrization, the layout of `sample_inputs` for one
example: heat_1d `b_1..b_5`, chloride_1d `[Cs, D]`, darcy_layered_1d `[k1, k2, x_s]`,
beam_1d (linear) `[q0, q1, EI]`, darcy_poisson_2d a 4 x 4 matrix, heat_2d a 3 x 3 matrix,
chloride_corner_2d `[Cs, D]`. Values outside the trained range are evaluated too but flagged
as **extrapolation**.

In [ ]:
import json
from pathlib import Path

from pino.inference import find_run, load_run, predict_one, save_prediction
from pino.visualization import plot_prediction

# The repository root, also when the notebook is started inside notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUTS = ROOT / "outputs"

## Available runs

In [ ]:
runs = []
for path in sorted(OUTPUTS.glob("*/*/*/*/metrics.json")):
    if "_superseded" in path.parts:
        continue
    m = json.loads(path.read_text())
    runs.append((m["benchmark"], f"{m.get('inputs')}/{m.get('output_param')}", m["mode"], m["seed"],
                 f"{100 * m['rel_l2']['mean']:.2f} %", str(path.parent.relative_to(ROOT))))
for row in runs:
    print("{:20s} {:22s} {:8s} seed {:3d}  rel. L2 {:>8s}  {}".format(*row))

## Load a run

In [ ]:
# Newest finished run of a combination; INPUTS_FAMILY None = the benchmark's original family.
BENCHMARK, MODE, OUTPUT_PARAM, INPUTS_FAMILY, SEED = "heat_1d", "physics", "constrained", None, None
RUN_DIR = find_run(BENCHMARK, MODE, OUTPUT_PARAM, INPUTS_FAMILY, SEED, outputs=OUTPUTS)
# RUN_DIR = ROOT / "outputs/heat_1d/sine-constrained/physics/<timestamp>"  # or a specific run

run = load_run(RUN_DIR)
low, high = run.problem.input_ranges()
print(RUN_DIR.relative_to(ROOT))
print(run.problem.name, run.mode, "| trained input ranges:")
print("  low ", low.tolist())
print("  high", high.tolist())

## Predict one input

Change `INPUTS` and `RESOLUTION` (a number or a list per axis, `None` = test grid) and run the cell again.

In [ ]:
INPUTS = [1.0, 0.0, 0.5, 0.0, 0.0]
RESOLUTION = None

prediction = predict_one(run, INPUTS, RESOLUTION)
print("rel. L2 ", {k: f"{100 * v:.3f} %" for k, v in prediction.rel_l2.items()})
print("bc error", {k: f"{100 * v:.3f} %" for k, v in prediction.bc_error.items()})
for message in prediction.extrapolation:
    print("EXTRAPOLATION:", message)
plot_prediction(run.problem, prediction, run.mode)  # last expression: shown inline

## Save

Writes `<run_dir>/predictions/<name>/` with `inputs.json`, `result.json`, `prediction.pt` and `solution.png`.

In [ ]:
# save_prediction(run, prediction, "my_case")